In [ ]:
import pandas as pd
import numpy as np
import pickle
from tqdm import tqdm

In [ ]:
import numpy as np

def R2_cal(y, y_pred):
    r2 = 1 - np.sum((y - y_pred)**2) / np.sum(y**2)
    return r2

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def sigmoid_grad(x):
    return sigmoid(x) * (1 - sigmoid(x))

def mean_squared_error(y, t):
    return 0.5 * np.sum((y - t) ** 2)

class TwoLayerNet:

    def __init__(self, input_size, hidden_size, output_size, weight_init_std=0.01):
        # 初始化权重
        self.params = {}
        self.params['W1'] = weight_init_std * np.random.randn(input_size, hidden_size)
        self.params['b1'] = np.zeros(hidden_size)
        self.params['W2'] = weight_init_std * np.random.randn(hidden_size, output_size)
        self.params['b2'] = np.zeros(output_size)

    def predict(self, x):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
    
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = a2  
        
        return y
        
    # x:输入数据, t:监督数据
    def loss(self, x, t):
        y = self.predict(x)
        return mean_squared_error(y, t)
    
    def accuracy(self, x, t):
        y = self.predict(x)
        error = np.mean((y - t) ** 2)  # 使用均方根误差(MSE)来衡量回归模型的性能
        
        return error
        
    # x:输入数据, t:监督数据   
    def gradient(self, x, t):
        W1, W2 = self.params['W1'], self.params['W2']
        b1, b2 = self.params['b1'], self.params['b2']
        grads = {}
        
        batch_num = x.shape[0]
        
        # forward
        a1 = np.dot(x, W1) + b1
        z1 = sigmoid(a1)
        a2 = np.dot(z1, W2) + b2
        y = a2
        
        # backward
        dy = (y - t) / batch_num
        grads['W2'] = np.dot(z1.T, dy)
        grads['b2'] = np.sum(dy, axis=0)
        
        da1 = np.dot(dy, W2.T)
        dz1 = sigmoid_grad(a1) * da1
        grads['W1'] = np.dot(x.T, dz1)
        grads['b1'] = np.sum(dz1, axis=0)

        return grads

In [ ]:
X = np.random.randn(1000,4)
y = 2 * X[:,0] + 4 * X[:,1] + 6 * X[:,2] + 8 * X[:,3]

In [ ]:
import time 

start_time = time.time()

x_train, y_train = X[:700], y[:700].reshape(-1, 1)
x_val, y_val = X[700:900], y[700:900].reshape(-1, 1)
x_test, y_test = X[900:], y[900:].reshape(-1, 1)

net_work = TwoLayerNet(input_size=4, hidden_size=16, output_size=1)

n_epochs = 100000
train_size = x_train.shape[0]
batch_size = 100
learning_rate = 0.02

patience = 20
best_model_state = None
best_loss = float('inf')
counter = 0

for epoch in range(n_epochs):
    for i in range(0, train_size, batch_size):
        xi = x_train[i:i + batch_size]
        yi = y_train[i:i + batch_size]

        grad = net_work.gradient(xi, yi)
        for key in ('W1', 'b1', 'W2', 'b2'):
            net_work.params[key] -= learning_rate * grad[key]
    
    val_loss = net_work.accuracy(x_val, y_val)
    
    if val_loss < best_loss:
        best_loss = val_loss
        best_model_state = net_work.params.copy()
        counter = 0
    else:
        counter += 1
        if counter >= patience:
            break

    if epoch % 2000 == 0:
        train_loss = net_work.accuracy(x_train, y_train)
        test_loss = net_work.accuracy(x_val, y_val)
        print(f"train loss: {train_loss}, val loss {val_loss}")

In [ ]:
for key in ('W1', 'b1', 'W2', 'b2'):
    net_work.params[key] = best_model_state[key]

In [ ]:
y_pred = net_work.predict(x_test)

In [ ]:
R2_cal(y_test, y_pred)